# Debugging a real pipeline: airline on-time performance

A quarter of a million flights, three joins, two Python UDFs, one grouped aggregate — and
one fault planted where it makes the answer look *almost* right.

Thirteen tools then take turns on it. Each answers a **different question**, and the point
is to watch the question narrow: from "something is wrong" to "this record, and this
branch of this function".

Every tool reports what it is about to do, what it found, and how long it took.

In [1]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

The reference tables are real: IATA carrier and airport codes with their cities and
states. The flights are generated — the actual BTS on-time files are hundreds of megabytes
a month — but generated to behave like the real thing, which is what matters here:

- **delays are heavy-tailed.** Most flights are near on time; a few are hours late. The
  tail is where the interesting bugs live, and a normal distribution would hide them.
- **carriers differ from each other**, and hubs differ from spokes, so a grouped aggregate
  has something to say.
- **some flights are cancelled** and carry no arrival time at all.
- **a few rows are malformed** the way a real feed is: a missing airport code, a delay
  recorded as `100000`.

Deterministic in the seed, so this notebook says the same thing twice.

In [2]:
import os, socket, sys, tempfile, time
sys.path.insert(0, f"{ROOT}/python")

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, StringType

import bigasterisk
from demos import airline_data

FLIGHTS = int(os.environ.get("FLIGHTS", "250000"))   # raise for a longer run
SEED = 7

# Where the work happens. In the Compose stack this is spark://master:7077 and the
# executors are in the worker containers; anywhere else it defaults to this machine.
MASTER = os.environ.get("BIGASTERISK_MASTER", "local[4]")

builder = (bigasterisk.configure(SparkSession.builder)
    .master(MASTER)
    .appName("airline analysis")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.skewJoin.enabled", "false"))

if MASTER.startswith("spark://"):
    # Client mode: the driver lives here and the executors dial it back, so it has to
    # advertise a name they can resolve rather than a container-local one.
    builder = (builder
        .config("spark.driver.host", os.environ.get("SPARK_DRIVER_HOST", socket.gethostname()))
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.executor.memory", os.environ.get("EXECUTOR_MEMORY", "2g"))
        .config("spark.cores.max", os.environ.get("CORES_MAX", "6")))
else:
    builder = builder.config("spark.ui.enabled", "false")

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "-", spark.sparkContext.master)

26/08/27 19:30:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.1.2 - spark://master:7077


In [3]:
started = time.time()
root = os.environ.get("AIRLINE_DATA",
                      os.path.join(tempfile.gettempdir(), "bigasterisk-airline-nb"))

flights = spark.createDataFrame(
    spark.sparkContext.parallelize(airline_data.flights(FLIGHTS, seed=SEED), 8),
    airline_data.FLIGHTS_SCHEMA)
airports = spark.createDataFrame(airline_data.airports(), airline_data.AIRPORTS_SCHEMA)
carriers = spark.createDataFrame(airline_data.carriers(), airline_data.CARRIERS_SCHEMA)

# Written out and read back: a file scan is what provenance capture attaches to, and it is
# also how the real feed arrives.
for name, df in (("flights", flights), ("airports", airports), ("carriers", carriers)):
    path = os.path.join(root, name)
    if not os.path.isdir(path):
        df.write.mode("overwrite").parquet(path)

flights = spark.read.parquet(os.path.join(root, "flights"))
airports = spark.read.parquet(os.path.join(root, "airports"))
carriers = spark.read.parquet(os.path.join(root, "carriers"))
for name, df in (("flights", flights), ("airports", airports), ("carriers", carriers)):
    df.createOrReplaceTempView(name)

print("%d flights, %d airports, %d carriers  (%.1fs)"
      % (flights.count(), airports.count(), carriers.count(), time.time() - started))
flights.show(5)

250000 flights, 30 airports, 10 carriers  (3.6s)
+---------+----------+-------+------+----+---------+---------+---------+--------+---------+
|flight_id|       day|carrier|origin|dest|sched_dep|dep_delay|arr_delay|distance|cancelled|
+---------+----------+-------+------+----+---------+---------+---------+--------+---------+
| F0124928|2026-09-23|     AS|   ATL| LAS|     1730|      -12|       -8|     234|        0|
| F0124929|2026-10-23|     UA|   CLT| DEN|     1845|       -6|      -11|    1292|        0|
| F0124930|2026-11-23|     AS|   CLT| ATL|      810|      -25|      -38|     767|        0|
| F0124931|2026-12-23|     AA|   MIA| SAN|     1030|        7|        7|    1123|        0|
| F0124932|2026-01-24|     B6|   ORD| ATL|     1605|       10|       -5|    2208|        0|
+---------+----------+-------+------+----+---------+---------+---------+--------+---------+
only showing top 5 rows


## Progress reporting

Every tool below runs inside this, so it says what it is doing rather than only what it
concluded. A tool that prints an answer and nothing else is impossible to trust.

In [4]:
class Step(object):
    """Narrates one tool: the question, the method, the findings, the time."""
    number = 0

    def __init__(self, tool, question, how):
        Step.number += 1
        self.tool, self.question, self.how = tool, question, how

    def __enter__(self):
        print("=" * 78)
        print("  %d. %s" % (Step.number, self.tool))
        print("     question: %s" % self.question)
        print("     method:   %s" % self.how)
        print("=" * 78)
        self.started = time.time()
        return self

    def say(self, line=""):
        print("     %s" % line)

    def finding(self, line):
        print("  >> %s" % line)

    def __exit__(self, kind, value, tb):
        print("     (%.1fs)" % (time.time() - self.started))
        return False


def flight(row):
    """One flight, readably. Witness rows carry only the columns a query touched."""
    if hasattr(row, "asDict"):
        row = row.asDict()
    if not isinstance(row, dict):
        return str(row)
    if row.get("flight_id") is None:
        return ", ".join("%s=%s" % (k, v) for k, v in row.items() if v is not None)
    return "%s %s %s->%s arr_delay=%s" % (row.get("flight_id"), row.get("carrier"),
                                          row.get("origin"), row.get("dest"),
                                          row.get("arr_delay"))

## The pipeline, and the fault

Two Python UDFs and three joins. `haul` classifies a route by distance. `adjusted_delay`
was meant to clamp implausible arrival delays and instead **flips their sign**.

It only fires above 300 minutes, so the vast majority of flights are untouched and the
aggregate looks nearly right. That is what makes it worth debugging rather than noticing.

In [5]:
def haul(distance):
    """Route length class."""
    if distance < 500:
        return "short"
    elif distance < 1500:
        return "medium"
    return "long"


def adjusted_delay(arr_delay):
    """Arrival delay, with extreme values 'corrected'."""
    if arr_delay is None:
        return None
    if arr_delay > 300:
        return -arr_delay          # <-- the fault
    return arr_delay


spark.udf.register("haul", haul, StringType())
spark.udf.register("adjusted_delay", adjusted_delay, IntegerType())

QUERY = """
    SELECT c.name AS carrier,
           haul(f.distance) AS haul,
           COUNT(*) AS flights,
           AVG(adjusted_delay(f.arr_delay)) AS avg_delay
    FROM flights f
    JOIN carriers c ON f.carrier = c.code
    JOIN airports o ON f.origin = o.code
    WHERE f.cancelled = 0
    GROUP BY c.name, haul(f.distance)
"""
ORACLE = "avg_delay < -20"

In [6]:
with Step("The pipeline", "what does the analysis say?",
          "three joins, two Python UDFs, one grouped aggregate") as step:
    rows = spark.sql(QUERY).orderBy("avg_delay").collect()
    step.say("%d (carrier, haul) groups" % len(rows))
    step.say("")
    step.say("%-24s %-8s %8s %10s" % ("carrier", "haul", "flights", "avg_delay"))
    for row in rows[:5]:
        step.say("%-24s %-8s %8d %10.1f"
                 % (row["carrier"], row["haul"], row["flights"], row["avg_delay"]))
    step.finding("the worst groups average large NEGATIVE delays — "
                 "no airline lands an hour early")

wrong = spark.sql(QUERY).filter(ORACLE).collect()
print("\n%d of %d groups are implausible (%s)" % (len(wrong), len(rows), ORACLE))

  1. The pipeline
     question: what does the analysis say?
     method:   three joins, two Python UDFs, one grouped aggregate


     30 (carrier, haul) groups
     
     carrier                  haul      flights  avg_delay
     Allegiant Air            short        2926      -92.9
     United Air Lines         long        11002      -46.9
     Delta Air Lines          medium      10677      -45.4
     Hawaiian Airlines        long        11116      -38.5
     Frontier Airlines        medium      10385      -37.0
  >> the worst groups average large NEGATIVE delays — no airline lands an hour early
     (2.1s)

12 of 30 groups are implausible (avg_delay < -20)


Something is wrong — but *what*? Seven operators and two functions over a quarter-million
rows. Each tool below narrows the question.

## 1. DeSQL — what is this query made of?

The query as a sequence of steps, each of which can be materialised on its own. This is
where the other tools get their notion of *an operation*.

In [7]:
with Step("DeSQL", "what are the parts of this query?",
          "decompose the plan; each step can be materialised on its own") as step:
    steps = bigasterisk.desql(spark).decompose(spark.sql(QUERY))
    for s in steps:
        step.say("[%d] %-12s %s" % (s.id, s.operator, (s.detail or "")[:58]))
    step.finding("%d steps, %d carrying conditional branches"
                 % (len(steps), sum(1 for s in steps if s.branches)))

  2. DeSQL
     question: what are the parts of this query?
     method:   decompose the plan; each step can be materialised on its own
     [0] Relation     flights AS f
     [1] Relation     carriers AS c
     [2] Join         INNER ON (f.carrier = c.code)
     [3] Relation     airports AS o
     [4] Join         INNER ON (f.origin = o.code)
     [5] Filter       (f.cancelled = 0)
     [6] Aggregate    c.name AS carrier, haul(distance) AS haul, count(1) AS fli
  >> 7 steps, 1 carrying conditional branches
     (0.1s)


## 2. Titian — which records produced the wrong number?

Record-level provenance, captured while the query runs and traced back to the source scan.
Exact — and, as you are about to see, not yet an answer.

In [8]:
# the worst group, deterministically: collect order is not stable, and every step after
# this one depends on which group we chase
worst = sorted(wrong, key=lambda r: r["avg_delay"])[0]
target = "carrier = '%s' AND haul = '%s'" % (worst["carrier"], worst["haul"])

with Step("Titian", "which input records produced %s?" % target,
          "record-level provenance captured during execution, traced to the scan") as step:
    lineage = bigasterisk.lineage(spark)
    step.say("enabling capture and re-running the query...")
    lineage.enable_capture()
    try:
        traced = spark.sql(QUERY)
        outputs = lineage.collect_with_lineage(traced)
        step.say("captured lineage for %d output groups" % len(outputs))
        picked = [(r, i) for r, i in outputs
                  if r["carrier"] == worst["carrier"] and r["haul"] == worst["haul"]]
        cursor = lineage.trace(traced, [picked[0][1]])
        hops = 0
        while not cursor.at_scan:
            cursor = cursor.go_back()
            hops += 1
        witnesses = cursor.show()
        step.say("walked back %d step(s) to the source scan" % hops)
        step.finding("%d source flights produced that one group" % len(witnesses))
        step.say("exact — and still far too many to read")
    finally:
        lineage.disable_capture()

  3. Titian
     question: which input records produced carrier = 'Allegiant Air' AND haul = 'short'?
     method:   record-level provenance captured during execution, traced to the scan
     enabling capture and re-running the query...
     captured lineage for 30 output groups


     walked back 3 step(s) to the source scan
  >> 2926 source flights produced that one group
     exact — and still far too many to read
     (2.2s)


## 3. FlowDebug — which of those records *mattered*?

Provenance says every record in the group contributed. Influence asks *how much*, reading
it from the aggregate's own semantics.

`via ...` is the taint refinement: of the flight's ten columns, only some could reach the
result at all.

In [9]:
with Step("FlowDebug", "of those flights, which one is responsible?",
          "influence from the aggregate's semantics, plus taint through the UDF") as step:
    influence_ranked = bigasterisk.influence(spark).influencers(
        spark.sql(QUERY), target, top_k=5)
    for inf in influence_ranked[:4]:
        step.say("%.4f  %s" % (inf.score, flight(inf.row)))
    step.finding("one flight carries %.1f%% of the responsibility"
                 % (influence_ranked[0].score * 100))
    step.say("columns that could reach the result: %s"
             % ", ".join(sorted(influence_ranked[0].columns)))

  4. FlowDebug
     question: of those flights, which one is responsible?
     method:   influence from the aggregate's semantics, plus taint through the UDF


     0.2686  F0126156 G4 LAX->SMF arr_delay=100000
     0.2686  F0139810 G4 EWR->ORD arr_delay=100000
     0.2686  F0247188 G4 SEA->MCO arr_delay=100000
     0.0017  F0229706 G4 SEA->PHL arr_delay=630
  >> one flight carries 26.9% of the responsibility
     columns that could reach the result: arr_delay
     (2.5s)


That is the malformed feed record — a delay of `100000` minutes.

## 4. Reading inside the UDFs

Everything so far treated `haul` and `adjusted_delay` as black boxes. Parsing their source
turns their branches into ordinary conditions over the columns the query passes them,
which is what the next tools need.

In [10]:
udf_profiles = {}

with Step("UDF analysis", "what is inside haul() and adjusted_delay()?",
          "parse the Python source: branches, paths, and which arguments matter") as step:
    for function in (haul, adjusted_delay):
        profile = bigasterisk.udf.register(spark, function)
        udf_profiles[profile.name] = profile
        step.say("%s" % profile)
        for condition, params in profile.branches:
            step.say("    branch: %s" % condition)
        for constraint, returns, exact in profile.paths:
            step.say("    path:   %-46s -> %s"
                     % (constraint, returns if returns else "(computed)"))
    step.finding("the UDFs are no longer black boxes")

  5. UDF analysis
     question: what is inside haul() and adjusted_delay()?
     method:   parse the Python source: branches, paths, and which arguments matter
     UdfProfile(haul(distance), 2 branches, 3 paths, solvable)
         branch: distance < 500
         branch: distance < 1500
         path:   distance < 500                                 -> 'short'
         path:   (NOT distance < 500) AND distance < 1500       -> 'medium'
         path:   (NOT distance < 500) AND (NOT distance < 1500) -> 'long'
     UdfProfile(adjusted_delay(arr_delay), 2 branches, 3 paths, solvable)
         branch: arr_delay IS NULL
         branch: arr_delay > 300
         path:   arr_delay IS NULL                              -> NULL
         path:   (NOT arr_delay IS NULL) AND arr_delay > 300    -> (computed)
         path:   (NOT arr_delay IS NULL) AND (NOT arr_delay > 300) -> (computed)
  >> the UDFs are no longer black boxes
     (0.0s)


## 5. OptDebug — which *operation* is wrong?

The tools above found the bad **data**. This one looks for bad **code**: every operation
and every branch scored by the records that reach it.

Watch for the UDF's own branches in the ranking — they are only there because the
functions were read in the previous step.

These tools re-execute the query many times, so they run on a sample rather than the whole
feed — and the sample is built deliberately: a clean slice, plus the single flight
influence named.

That matters more than it looks. Put a malformed record into every group and every group
is faulty, so there is nothing for a spectrum to discriminate against and every operation
ties at the top. A ranking is only informative when some records pass.

The other half is **minimisation**. `arr_delay > 300` is taken by thousands of
legitimately late flights as well as by the malformed one, so scoring it directly says
little. Narrowing the failing input to the records that actually cause the failure — delta
debugging, before any scoring — is what makes the branch stand out.

In [11]:
culprit_id = influence_ranked[0].row["flight_id"]
scoped_path = os.path.join(root, "flights_scoped")
clean = spark.table("flights").filter(
    # every twentieth flight, chosen by its id rather than by sampling: `sample()` depends
    # on how the file splits happen to be assigned, so it is not reproducible across runs
    "(arr_delay IS NULL OR arr_delay < 10000) AND CAST(substring(flight_id, 2) AS INT) % 20 = 0")
scoped = clean.unionByName(
    spark.table("flights").filter("flight_id = '%s'" % culprit_id))
scoped.write.mode("overwrite").parquet(scoped_path)
spark.read.parquet(scoped_path).createOrReplaceTempView("flights")
scoped_count = spark.table("flights").count()


def operation(op):
    where = op.branch if op.branch else (op.detail or "")
    return "[%s] %s %s" % (op.step_id, op.operator, where[:66])


with Step("OptDebug", "which operation produces the wrong number?",
          "score every operation and branch by the records that reach it") as step:
    step.say("a clean %d-flight sample, plus the one flight influence named (%s)"
             % (scoped_count, culprit_id))
    # base_table narrows the failing records to those that actually cause the failure
    # *before* scoring. Without it the branch `arr_delay > 300` is taken by thousands of
    # legitimately late flights as well as by the malformed one, and a spectrum cannot
    # separate them.
    optdebug_result = bigasterisk.optdebug(spark).localize(
        QUERY, ORACLE, base_table="flights")
    if optdebug_result.minimised:
        step.say("narrowed the failing input from %d records before scoring"
                 % optdebug_result.minimised_from)
    for op in optdebug_result.ranked[:8]:
        step.say("%.3f  %s" % (op.score, operation(op)))
    inside = [(i, op) for i, op in enumerate(optdebug_result.ranked, 1)
              if op.branch and ("arr_delay" in op.branch or "distance" in op.branch)]
    rank, op = inside[0]
    step.finding("highest-ranked branch from inside a UDF is #%d: %s (%.3f)"
                 % (rank, op.branch, op.score))
    step.say("those branches are invisible to plan analysis; they are here only "
             "because the functions were read")

  6. OptDebug
     question: which operation produces the wrong number?
     method:   score every operation and branch by the records that reach it
     a clean 12499-flight sample, plus the one flight influence named (F0126156)
     narrowed the failing input from 124 records before scoring
     0.989  [6] Aggregate (f.arr_delay > 300)
     0.902  [6] Aggregate (f.distance < 500)
     0.656  [6] Aggregate (f.distance < 1500)
     0.505  [5] Filter (f.cancelled = 0)
     0.505  [2] Join INNER ON (f.carrier = c.code)
     0.505  [4] Join INNER ON (f.origin = o.code)
     0.505  [5] Filter (f.cancelled = 0)
     0.505  [6] Aggregate c.name AS carrier, haul(distance) AS haul, count(1) AS flights, av
  >> highest-ranked branch from inside a UDF is #1: (f.arr_delay > 300) (0.989)
     those branches are invisible to plan analysis; they are here only because the functions were read
     (8.5s)


`arr_delay > 300` tops the ranking: the sign flip itself, named as an *operation* rather
than as a record — and a branch that lives inside a Python function, invisible to plan
analysis until that function was read.

Spectrum-based ranking narrows a search; it does not hand you a line number. What it did
here was put the faulty branch above every operator Spark's own plan contains.

## 6. BigSift — the smallest input that still fails

Delta debugging over the provenance, re-running the query on subsets until nothing more
can be removed.

In [12]:
with Step("BigSift", "what is the smallest input that still fails?",
          "delta debugging over the provenance, re-running to check each subset") as step:
    step.say("starting from %d flights..." % scoped_count)
    bigsift_result = bigasterisk.BigSift(spark).debug(
        "flights", QUERY,
        lambda row: row["avg_delay"] is not None and row["avg_delay"] < -20)
    step.say("provenance left %d candidate records" % bigsift_result.provenance_size)
    for row in bigsift_result.fault_inducing_rows[:3]:
        step.say("    %s" % flight(row))
    step.finding("%d record(s) reproduce the failure on their own"
                 % len(bigsift_result.fault_inducing_rows))

spark.read.parquet(os.path.join(root, "flights")).createOrReplaceTempView("flights")

  7. BigSift
     question: what is the smallest input that still fails?
     method:   delta debugging over the provenance, re-running to check each subset
     starting from 12499 flights...


     provenance left 124 candidate records
         carrier=G4, origin=LAX, arr_delay=100000, distance=231, cancelled=0
  >> 1 record(s) reproduce the failure on their own
     (4.8s)


## 7. BigDebug — watch the feed, and survive the crash

Two primitives: a watchpoint that captures records matching a condition as they flow past,
and a guard that names the exact record that killed a task — partition and index — instead
of a stack trace.

In [13]:
def quieten(loggers, level="OFF"):
    """Silence named JVM loggers around an expected failure."""
    try:
        jvm = spark._jvm
        configurator = jvm.org.apache.logging.log4j.core.config.Configurator
        target = getattr(jvm.org.apache.logging.log4j.Level, level)
        for name in loggers:
            configurator.setLevel(name, target)
    except Exception:
        pass


NOISY = ["SQLQueryContextLogger", "org.apache.spark.scheduler.TaskSetManager"]

with Step("BigDebug", "can I watch the feed and survive a crash?",
          "a watchpoint on the records flowing past, and a guard that names the "
          "record that kills a task") as step:
    watched = bigasterisk.watchpoints(spark).watch(
        spark.table("flights"), col("arr_delay") > 10000)
    watched.df.createOrReplaceTempView("flights")
    spark.sql(QUERY).collect()
    step.say("watchpoint matched %d record(s)" % watched.hits)
    for row in watched.captured[:3]:
        step.say("    %s" % flight(row))
    spark.read.parquet(os.path.join(root, "flights")).createOrReplaceTempView("flights")

    step.say("")
    step.say("now a query that divides by the distance from that outlier...")
    guard = bigasterisk.crash_culprit(spark).guard(
        spark.table("flights").filter("arr_delay IS NOT NULL").coalesce(1))
    previous = spark.conf.get("spark.sql.ansi.enabled")
    spark.conf.set("spark.sql.ansi.enabled", "true")
    quieten(NOISY)
    spark.sparkContext.setLogLevel("OFF")
    try:
        guard.df.selectExpr("flight_id", "100 DIV (arr_delay - 100000) AS boom").collect()
        step.say("expected a failure and did not get one")
    except Exception:
        culprit = guard.culprit
        step.finding("the record that killed the task: %s" % flight(culprit.row))
        step.say("partition %d, record %d" % (culprit.partition_id, culprit.record_index))
    finally:
        spark.sparkContext.setLogLevel("ERROR")
        quieten(NOISY, "ERROR")
        spark.conf.set("spark.sql.ansi.enabled", previous)

  8. BigDebug
     question: can I watch the feed and survive a crash?
     method:   a watchpoint on the records flowing past, and a guard that names the record that kills a task
     watchpoint matched 70 record(s)
         F0188979 B6 MIA->ATL arr_delay=100000
         F0189166 HA DFW->LAS arr_delay=100000
         F0194577 G4 PHX->MSP arr_delay=100000
     
     now a query that divides by the distance from that outlier...


{"ts": "2026-08-27 19:38:07.391", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set \"spark.sql.ansi.enabled\" to \"false\" to bypass this error. SQLSTATE: 22012", "context": {"errorClass": "DIVIDE_BY_ZERO"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o506.collectToPython.\n: org.apache.spark.SparkArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set \"spark.sql.ansi.enabled\" to \"false\" to bypass this error. SQLSTATE: 22012\n== SQL (line 1, position 1) ==\n100 DIV (arr_delay - 100000) AS boom\n^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.divideByZeroError(QueryExecutionErrors.scala:205)\n\tat org.apache.spark.sql.errors.QueryExecutionErrors.divideByZeroError(QueryExecutionErrors.scala)\n

  >> the record that killed the task: F0126156 G4 LAX->SMF arr_delay=100000
     partition 0, record 1205
     (1.2s)


## 8. PerfDebug — where does the time go?

In [14]:
with Step("PerfDebug", "which records cost the most to process?",
          "per-record latency, attributed back to the input rows") as step:
    sample = spark.table("flights").limit(20000).coalesce(1)
    profile = bigasterisk.perfdebug(spark).profile(sample, top_k=3)
    profile.df.createOrReplaceTempView("flights")
    spark.sql(QUERY).collect()
    step.say("%d records profiled, skew %.1fx the mean" % (profile.records, profile.skew))
    for cost in profile.slowest[:3]:
        step.say("    %.3f ms  %s" % (cost.millis, flight(cost.row)))
    step.finding("at this scale the top record is task warm-up rather than data; "
                 "the attribution is what is being shown")

spark.read.parquet(os.path.join(root, "flights")).createOrReplaceTempView("flights")

  9. PerfDebug
     question: which records cost the most to process?
     method:   per-record latency, attributed back to the input rows
     19999 records profiled, skew 261.9x the mean
         0.295 ms  F0000058 NK MSP->PDX arr_delay=-10
         0.291 ms  F0000060 WN MCO->ORD arr_delay=-26
         0.118 ms  F0000586 HA BWI->SEA arr_delay=10
  >> at this scale the top record is task warm-up rather than data; the attribution is what is being shown
     (0.4s)


## 9. Vega — how much can the fix reuse?

You edit the query and run it again. Vega reuses what the edit did not invalidate.

In [15]:
with Step("Vega", "how much of the fix can reuse the last run?",
          "match the revised plan against what the previous run materialised") as step:
    incremental = bigasterisk.vega(spark)
    try:
        step.say("running the original...")
        incremental.run(spark.sql(QUERY)).df.collect()
        fixed = QUERY.replace("WHERE f.cancelled = 0",
                              "WHERE f.cancelled = 0 AND f.arr_delay < 10000")
        step.say("running the fix (a guard against the malformed feed rows)...")
        revised = incremental.run(spark.sql(fixed))
        revised.df.collect()
        step.finding("reused %d of %d parts (%.0f%%)"
                     % (len(revised.reused), revised.steps, revised.reuse_ratio * 100))
        for part in revised.reused[:3]:
            step.say("    reused: %s" % part[:70])
    finally:
        incremental.clear()

  10. Vega
     question: how much of the fix can reuse the last run?
     method:   match the revised plan against what the previous run materialised
     running the original...
     running the fix (a guard against the malformed feed rows)...
  >> reused 2 of 7 parts (29%)
         reused: Join — INNER ON (f.carrier = c.code)
         reused: Join — INNER ON (f.origin = o.code)
     (2.4s)


## 10. BigTest and NaturalSym — an input per path, through the UDF

Fuzzing searches for inputs; this constructs them. `haul(distance) = 'long'` cannot be
inverted as written — but the conditions under which `haul` *returns* `'long'` are ordinary
comparisons on `distance`, and those can be solved.

Compare the two generated records below. Same path; very different readability. That is
NaturalSym's contribution.

In [16]:
UDF_QUERY = "SELECT flight_id FROM flights WHERE haul(distance) = 'long'"

with Step("BigTest", "what input drives each path, including inside the UDF?",
          "solve the query's branch conditions; the UDF's paths are now among them") as step:
    suite = bigasterisk.testgen(spark).generate(
        UDF_QUERY, {"flights": spark.table("flights")},
        rows_per_path=1, natural=False, seed=5)
    for case in suite.cases:
        step.say("%s" % case)
    step.finding("%d of %d verified; the condition on the UDF's result became a "
                 "condition on `distance`" % (len(suite.verified), len(suite.cases)))

with Step("NaturalSym", "the same paths, with values that look real",
          "witnesses drawn from values that occur, shaped by a declared distribution") as step:
    natural = bigasterisk.testgen(spark).generate(
        UDF_QUERY, {"flights": spark.table("flights")}, rows_per_path=1, natural=True,
        seed=5, distributions={"distance": "normal(1200, 600)"})
    for case in natural.cases:
        step.say("%s" % case)
    step.finding("same paths, records that read like records")

  11. BigTest
     question: what input drives each path, including inside the UDF?
     method:   solve the query's branch conditions; the UDF's paths are now among them
     [--] NOT ((NOT (flights.distance < 500)) AND (NOT (flights.distance < 1500)))  (condition outside the solver's fragment)

     [ok] ((NOT (flights.distance < 500)) AND (NOT (flights.distance < 1500)))  (verified)
    flights: [78gU6H,SFwnpt,xq6AI7,FPQRYk,NocJZC,89,92,57,1500,29]
  >> 1 of 2 verified; the condition on the UDF's result became a condition on `distance`
     (0.2s)
  12. NaturalSym
     question: the same paths, with values that look real
     method:   witnesses drawn from values that occur, shaped by a declared distribution
     [--] NOT ((NOT (flights.distance < 500)) AND (NOT (flights.distance < 1500)))  (condition outside the solver's fragment)

     [ok] ((NOT (flights.distance < 500)) AND (NOT (flights.distance < 1500)))  (verified)
    flights: [F0125415,2026-01-10,DL,DTW,CLT,1030,409,294,229

## 11. BigFuzz, DepFuzz and NaturalFuzz — what else would break it?

One query, three mutation strategies. The three papers differ in exactly one decision —
where a generated value comes from — and on a joined query the difference is stark.

In [19]:
fuzz_empty = {}

with Step("BigFuzz / DepFuzz / NaturalFuzz", "what else would break this pipeline?",
          "one query, three mutation strategies") as step:
    seeds = {"flights": spark.table("flights").limit(2000),
             "carriers": spark.table("carriers")}
    FUZZ_QUERY = ("SELECT c.name, COUNT(*) AS n FROM flights f "
                  "JOIN carriers c ON f.carrier = c.code "
                  "WHERE f.arr_delay > 60 GROUP BY c.name")
    step.say("%-14s %10s %16s %9s" % ("strategy", "coverage", "empty results", "crashes"))
    for strategy in ("random", "natural", "co-dependent"):
        outcome = bigasterisk.fuzz(spark).fuzz(
            FUZZ_QUERY, seeds, iterations=200000, strategy=strategy, seed=1)
        fuzz_empty[strategy] = outcome.empty_results
        step.say("%-14s %9.0f%% %12d/%-3d %9d"
                 % (strategy, outcome.coverage * 100, outcome.empty_results,
                    outcome.iterations, len(outcome.failures)))
    step.finding("a join key invented from nothing rarely matches, so most of the "
                 "random campaign is wasted; repairing the equality fixes it")

  15. BigFuzz / DepFuzz / NaturalFuzz
     question: what else would break this pipeline?
     method:   one query, three mutation strategies
     strategy         coverage    empty results   crashes
     random               100%       185287/200000         0
     natural              100%       115836/200000         0
     co-dependent         100%         3972/200000         0
  >> a join key invented from nothing rarely matches, so most of the random campaign is wasted; repairing the equality fixes it
     (52.5s)


## What each tool answered

| Tool | Question | Answer |
|---|---|---|
| DeSQL | what are the parts? | 7 steps, 1 with branches |
| Titian | which records? | thousands of source flights — exact, unreadable |
| FlowDebug | which of them mattered? | one flight, and only its `arr_delay` |
| UDF analysis | what is inside the functions? | 2 branches, 3 paths each |
| OptDebug | which operation? | the two UDF branches top the ranking; one is the fault |
| BigSift | smallest failing input? | 1 record |
| BigDebug | watch and survive? | the malformed records; the record that killed the task |
| PerfDebug | where did time go? | per-record cost attribution |
| Vega | can the fix reuse anything? | both joins |
| BigTest | an input per path? | solved through the UDF |
| NaturalSym | ...that looks real? | a plausible flight record |
| BigFuzz / DepFuzz / NaturalFuzz | what else breaks it? | random wastes the campaign; co-dependent does not |

**Two different faults, found by two different kinds of tool.** A malformed record in the
feed (Titian, FlowDebug, BigSift) and a wrong branch in a UDF (OptDebug). Neither class of
tool would have found the other's.

## Check

This notebook runs in CI, so it ends in assertions rather than prose.

In [ ]:
assert len(wrong) > 0, "the planted fault should make some groups implausible"

# provenance is exact but wide; influence narrows it, and taint narrows the columns
assert influence_ranked, "influence should rank something"
assert influence_ranked[0].columns == {"arr_delay"}, influence_ranked[0].columns

# the UDFs were read
assert udf_profiles["haul"].solvable
assert udf_profiles["adjusted_delay"].solvable
assert len(udf_profiles["adjusted_delay"].branches) == 2

# Operation-level localisation reaches a branch inside a UDF. What the analysis
# guarantees is that the branch is *visible* — where it ranks depends on how sharply the
# failing and passing populations differ, which is a property of the data.
branches = [op.branch for op in optdebug_result.ranked if op.branch]
assert any("arr_delay" in b for b in branches), branches
assert any("distance" in b for b in branches), branches

# input isolation reaches a handful of records
assert 0 < len(bigsift_result.fault_inducing_rows) <= 3

# and the fuzzers disagree the way the papers say they do
assert fuzz_empty["random"] > fuzz_empty["co-dependent"], fuzz_empty
print("OK - every tool answered its question")